In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("titanic_data_updated.csv")

In [2]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
697,698,yes,third,"Mullens, Miss. Katherine ""Katie""",female,NaN,0,0,35852,7.7333,NaN,Q
197,198,no,third,"Olsen, Mr. Karl Siegwart Andreas",male,42.0,0,1,4579,8.4042,NaN,S
890,891,no,third,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.7500,NaN,Q
448,449,yes,third,"Baclini, Miss. Marie Catherine",female,5.0,2,1,2666,19.2583,NaN,C
577,578,yes,first,"Silvey, Mrs. William Baird (Alice Munger)",female,39.0,1,0,13507,55.9000,E44,S


Deleting useless columns

In [3]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

Combining 'SibSp' and 'Parch' columns

In [4]:
df['FamilyMembers'] = df['SibSp'] + df['Parch'] + 1 # 1 is for the person him/herself

In [5]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [6]:
df.sample(2)

,Survived,Pclass,Sex,Age,Fare,Cabin,Embarked,FamilyMembers
498,no,first,female,25.0,151.550,C22 C26,S,4
62,no,first,male,45.0,83.475,C83,S,2


Train Test Split

In [7]:
x = df.drop(columns='Survived') # features
y = df['Survived'] # target

x_train, x_test, y_train, y_test = train_test_split(x,y,train_size=0.2, random_state=42)

Imputation

In [9]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan,strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan,strategy='most_frequent'),['Embarked']),
        ('cabin',SimpleImputer(missing_values=np.nan,strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

imputer_transformer.set_output(transform='pandas')

# fit
imputer_transformer.fit(x_train)

# transform
x_train = imputer_transformer.transform(x_train)
x_test = imputer_transformer.transform(x_test)

In [10]:
x_train.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [11]:
x_test.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [15]:
x_train.sort_values(by='Age').tail(10)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age
698,49.0,C,C68,False,first,male,110.8833,3,1.611659
52,49.0,C,D33,False,first,female,76.7292,2,1.611659
458,50.0,S,Missing,True,second,female,10.5000,1,1.698099
406,51.0,S,Missing,True,third,male,7.7500,1,1.784539
492,55.0,S,C30,False,first,male,30.5000,1,2.130298
647,56.0,C,A26,False,first,male,35.5000,1,2.216738
366,60.0,C,D37,False,first,female,75.2500,2,2.562498
170,61.0,S,B19,False,first,male,33.5000,1,2.648938
555,62.0,S,Missing,True,first,male,26.5500,1,2.735378
252,62.0,S,C87,False,first,male,26.5500,1,2.735378


### Outliers Handling

Age Outliers

In [13]:
mean_age = x_train['Age'].mean()
std_age = x_train['Age'].std()

x_train['z_score_age'] = (x_train['Age']-mean_age)/std_age

outliers_age = x_train[abs(x_train['z_score_age'])>3]

print(f"Number of outliers: {len(outliers_age)}")
# display(outliers_age)
# x_train.sort_values(by='Age').tail(10)

Number of outliers: 0


Fare outliers

In [16]:
mean_fare = x_train['Fare'].mean()
std_fare = x_train['Fare'].std()

x_train['z_score_fare'] = (x_train['Fare']-mean_fare)/std_fare
outliers_fare = x_train[abs(x_train['z_score_fare'])>3]

print(f"Number of outlier fare: {len(outliers_fare)}")
display(outliers_fare.head())

Number of outlier fare: 3


,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
27,19.0,S,C23 C25 C27,False,first,male,263.000,6,-0.981541,6.267455
498,25.0,S,C22 C26,False,first,female,151.550,4,-0.462901,3.280644
700,18.0,C,C62 C64,False,first,female,227.525,2,-1.067981,5.316741


Fare outliers using IQR

In [17]:
fare_q1 = x_train['Fare'].quantile(0.25)
fare_q3 = x_train['Fare'].quantile(0.75)

fare_iqr = fare_q3 - fare_q1

max_range = fare_q3 + (1.5 * fare_iqr)
min_range = max(0,fare_q1 - (1.5 * fare_iqr)) # becasue the value goes to negative side, so capping at 0

print(f"Max: {max_range}\nMin: {min_range}")

fare_outliers = x_train[(x_train['Fare']<min_range) | (x_train['Fare'] > max_range)]
print("Number of fare outliers: ",len(fare_outliers))
display(fare_outliers.sample(5))

Max: 65.3438
Min: 0
Number of fare outliers:  24


,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
698,49.0,C,C68,False,first,male,110.8833,3,1.611659,2.190794
366,60.0,C,D37,False,first,female,75.2500,2,2.562498,1.235837
102,21.0,S,D26,False,first,male,77.2875,2,-0.808661,1.290441
681,27.0,C,D49,False,first,male,76.7292,1,-0.290021,1.275479
1,38.0,C,C85,False,first,female,71.2833,2,0.660819,1.129531


In [18]:
# omitting outliers age from the x_train data
x_train = x_train[abs(x_train['z_score_age']) <=3]
x_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
761,41.000000,S,Missing,True,third,male,7.1250,1,0.920139,-0.589883
645,48.000000,C,D33,False,first,male,76.7292,2,1.525219,1.275479
754,48.000000,S,Missing,True,second,female,65.0000,4,1.525219,0.961141
556,48.000000,C,A16,False,first,female,39.6000,2,1.525219,0.280433
850,4.000000,S,Missing,True,third,male,31.2750,7,-2.278141,0.057326
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,7.6500,1,-0.808661,-0.575814
270,30.355172,S,Missing,True,first,male,31.0000,1,0.000000,0.049956
860,41.000000,S,Missing,True,third,male,14.1083,3,0.920139,-0.402734
435,14.000000,S,B96 B98,False,first,female,120.0000,4,-1.413741,2.435117


In [19]:
# Capping 'Fare' at 0 to max range of IQR outliers
x_train['Fare'] = x_train['Fare'].clip(min_range,max_range)
print(x_train['Fare'].min())
print(x_train['Fare'].max())

0.0
65.3438


In [20]:
# x_train.sort_values(by='Age').tail(5)
x_train.sort_values(by='Fare').tail(5)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
681,27.0,C,D49,False,first,male,65.3438,1,-0.290021,1.275479
385,18.0,S,Missing,True,second,male,65.3438,1,-1.067981,1.188938
700,18.0,C,C62 C64,False,first,female,65.3438,2,-1.067981,5.316741
435,14.0,S,B96 B98,False,first,female,65.3438,4,-1.413741,2.435117
102,21.0,S,D26,False,first,male,65.3438,2,-0.808661,1.290441
